# 05 - Estimate the match days
**What it does:** <in your own words, one or two sentences>
**Needs:** notebook 04 has run (tennis_clean.matches exists)
**Produces:** the table tennis_clean.matches_dated: each match with an event format and a window of possible match days
**Run after:** 04_clean_matches

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
CLEAN = f"{PROJECT_ID}.tennis_clean.matches"
DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"

## Estimate a window of possible match days
Only the tournament start date is known, so each match gets a range of possible days (`match_date_min` to `match_date_max`), not a single day. The weather features later average over that range, and the weather fetch (notebook 08) covers the same days.

The event format comes from the first round present in the data: Grand Slams (level G), two-week events (first round R128), longer one-week events (first round R64) and ordinary one-week events. The day offsets per round in `WINDOWS` are assumptions based on typical schedules (wide on purpose), kept in code so they can be changed and rebuilt.

In [ ]:
# Day offsets from tourney_date: (earliest, latest). Typical-schedule assumptions, wide on purpose.
WINDOWS = {
    "SLAM":      {"R128": (0, 1), "R64": (2, 3), "R32": (4, 5), "R16": (6, 7), "QF": (8, 9), "SF": (9, 11), "F": (11, 13)},
    "TWO_WEEK":  {"R128": (0, 4), "R64": (2, 6), "R32": (4, 8), "R16": (7, 10), "QF": (8, 11), "SF": (9, 12), "F": (11, 13)},
    "LONG_WEEK": {"R64": (0, 2), "R32": (1, 3), "R16": (2, 4), "QF": (3, 5), "SF": (4, 6), "F": (5, 7)},
    "ONE_WEEK":  {"R32": (0, 2), "R16": (1, 3), "QF": (2, 4), "SF": (3, 5), "F": (5, 6)},
}

rows = []
for fmt, rounds in WINDOWS.items():
    for rnd, (lo, hi) in rounds.items():
        rows.append(f"SELECT '{fmt}' AS fmt, '{rnd}' AS round, {lo} AS day_min, {hi} AS day_max")
windows_sql = "\nUNION ALL\n".join(rows)

sql = f"""
CREATE OR REPLACE TABLE `{DATED}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'tennis_clean.matches plus estimated match-day windows (event format and round).')
AS
WITH event_format AS (
  SELECT tourney_id,
    CASE
      WHEN ANY_VALUE(tourney_level) = 'G' THEN 'SLAM'
      WHEN LOGICAL_OR(round = 'R128') THEN 'TWO_WEEK'
      WHEN LOGICAL_OR(round = 'R64') THEN 'LONG_WEEK'
      ELSE 'ONE_WEEK'
    END AS fmt
  FROM `{CLEAN}` GROUP BY tourney_id
),
windows AS ({windows_sql})
SELECT m.*, f.fmt AS event_format, w.day_min, w.day_max,
       DATE_ADD(m.tourney_date, INTERVAL w.day_min DAY) AS match_date_min,
       DATE_ADD(m.tourney_date, INTERVAL w.day_max DAY) AS match_date_max
FROM `{CLEAN}` m
JOIN event_format f USING (tourney_id)
LEFT JOIN windows w ON w.fmt = f.fmt AND w.round = m.round
"""
job = bq.query(sql); job.result()
print("Built", DATED, "| bytes processed:", f"{job.total_bytes_processed:,}")

## Checks
1. Same number of rows as the clean table, and every match got a window (`no_window` is 0).
2. Which weekday do tournaments start on? Assumed to be mostly Monday. Events starting on other days (mostly 2020 and 2021) may have windows about a day off.
3. Sanity check against known dates: the 2019 women's Grand Slam finals were played on 26 Jan, 8 Jun, 13 Jul and 7 Sep. Each must fall inside its window. This is a sanity check of four matches, not a full validation.

In [ ]:
# 1. Every match has a window
counts = run(f"""SELECT (SELECT COUNT(*) FROM `{CLEAN}`) AS clean_rows, (SELECT COUNT(*) FROM `{DATED}`) AS dated_rows""")
print(counts)
assert counts["clean_rows"][0] == counts["dated_rows"][0]
fmt = run(f"""SELECT event_format, COUNT(DISTINCT tourney_id) AS events, COUNT(*) AS matches,
                     COUNTIF(day_min IS NULL) AS no_window FROM `{DATED}` GROUP BY 1 ORDER BY 1""")
print(fmt)
assert fmt["no_window"].sum() == 0, "Some matches have no window: a round label is not in WINDOWS"

# 2. Start weekdays
print(run(f"""SELECT FORMAT_DATE('%a', tourney_date) AS weekday, COUNT(DISTINCT tourney_id) AS events
              FROM `{DATED}` GROUP BY 1 ORDER BY events DESC"""))

# 3. 2019 Slam finals against their windows
run(f"""SELECT name_key, tourney_date, FORMAT_DATE('%a', tourney_date) AS start_day, match_date_min, match_date_max
        FROM `{DATED}` WHERE season = 2019 AND tourney_level = 'G' AND round = 'F' ORDER BY tourney_date""")